# 🅿️ Day 01a: LLD — Parking Lot System

---

## 🎯 What You'll Master Today
- Requirements gathering for LLD
- Class diagram design
- Vehicle/Spot/ParkingLot hierarchy
- Payment system integration
- Complete OOP implementation

**The classic LLD interview question.** Show clean OOP, SOLID principles, and extensibility.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  PARKING LOT — REQUIREMENTS                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Functional:                                                     ║
║  • Multiple floors, each with multiple spots                    ║
║  • Vehicle types: Motorcycle, Car, Truck                        ║
║  • Spot types: Small, Medium, Large                             ║
║  • Motorcycle → Small, Car → Medium, Truck → Large              ║
║  • Park vehicle, unpark vehicle                                  ║
║  • Check availability per floor/type                             ║
║  • Generate ticket on entry, calculate fee on exit              ║
║                                                                   ║
║  Non-functional:                                                 ║
║  • Thread-safe (concurrent entry/exit)                           ║
║  • O(1) find available spot                                      ║
║                                                                   ║
║  Class Diagram:                                                  ║
║  ┌──────────┐    has    ┌───────┐    has    ┌──────────┐        ║
║  │ParkingLot│──────────►│ Floor │──────────►│   Spot   │        ║
║  └──────────┘           └───────┘           └──────────┘        ║
║       │                                         ▲                ║
║       │ issues                    occupied_by   │                ║
║  ┌──────────┐              ┌──────────┐        │                ║
║  │  Ticket  │              │ Vehicle  │────────┘                ║
║  └──────────┘              └──────────┘                         ║
║                                 ▲                                ║
║                    ┌────────────┼────────────┐                  ║
║                    │            │            │                   ║
║               Motorcycle      Car         Truck                 ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Enums and Vehicle Classes
# ============================================================

from enum import Enum
from datetime import datetime
import threading

class VehicleType(Enum):
    MOTORCYCLE = 1
    CAR = 2
    TRUCK = 3

class SpotSize(Enum):
    SMALL = 1   # motorcycle
    MEDIUM = 2  # car
    LARGE = 3   # truck

# Vehicle type → minimum spot size needed
VEHICLE_TO_SPOT = {
    VehicleType.MOTORCYCLE: SpotSize.SMALL,
    VehicleType.CAR: SpotSize.MEDIUM,
    VehicleType.TRUCK: SpotSize.LARGE,
}

class Vehicle:
    def __init__(self, license_plate: str, vehicle_type: VehicleType):
        self.license_plate = license_plate
        self.vehicle_type = vehicle_type
    
    def __repr__(self):
        return f"{self.vehicle_type.name}({self.license_plate})"

# Convenience constructors
class Motorcycle(Vehicle):
    def __init__(self, plate): super().__init__(plate, VehicleType.MOTORCYCLE)

class Car(Vehicle):
    def __init__(self, plate): super().__init__(plate, VehicleType.CAR)

class Truck(Vehicle):
    def __init__(self, plate): super().__init__(plate, VehicleType.TRUCK)

print("✅ Vehicle hierarchy defined")
print(Car("KA-01-1234"))

In [ ]:
# ============================================================
# 2. Spot and Floor
# ============================================================

class ParkingSpot:
    def __init__(self, spot_id: int, size: SpotSize, floor: int):
        self.spot_id = spot_id
        self.size = size
        self.floor = floor
        self.vehicle = None  # None = available
    
    @property
    def is_available(self):
        return self.vehicle is None
    
    def can_fit(self, vehicle: Vehicle) -> bool:
        required = VEHICLE_TO_SPOT[vehicle.vehicle_type]
        return self.is_available and self.size.value >= required.value
    
    def park(self, vehicle: Vehicle):
        if not self.can_fit(vehicle):
            raise ValueError(f"Cannot park {vehicle} in {self.size.name} spot")
        self.vehicle = vehicle
    
    def unpark(self) -> Vehicle:
        v = self.vehicle
        self.vehicle = None
        return v
    
    def __repr__(self):
        status = self.vehicle or "empty"
        return f"Spot(F{self.floor}-{self.spot_id}, {self.size.name}, {status})"


class ParkingFloor:
    def __init__(self, floor_num: int, small=5, medium=10, large=3):
        self.floor_num = floor_num
        self.spots = []
        spot_id = 1
        for size, count in [(SpotSize.SMALL, small), (SpotSize.MEDIUM, medium), (SpotSize.LARGE, large)]:
            for _ in range(count):
                self.spots.append(ParkingSpot(spot_id, size, floor_num))
                spot_id += 1
    
    def find_spot(self, vehicle: Vehicle):
        """Find first available spot that fits the vehicle."""
        for spot in self.spots:
            if spot.can_fit(vehicle):
                return spot
        return None
    
    def available_count(self):
        return {size.name: sum(1 for s in self.spots if s.size == size and s.is_available)
                for size in SpotSize}

print("✅ Spot and Floor defined")
floor = ParkingFloor(1)
print(f"Floor 1 availability: {floor.available_count()}")

In [ ]:
# ============================================================
# 3. Ticket and ParkingLot (Main Class)
# ============================================================

class Ticket:
    _counter = 0
    
    def __init__(self, vehicle: Vehicle, spot: ParkingSpot):
        Ticket._counter += 1
        self.ticket_id = Ticket._counter
        self.vehicle = vehicle
        self.spot = spot
        self.entry_time = datetime.now()
        self.exit_time = None
    
    def __repr__(self):
        return f"Ticket#{self.ticket_id}({self.vehicle}, {self.spot})"


class ParkingLot:
    """Thread-safe parking lot with multiple floors."""
    
    # Hourly rates
    RATES = {VehicleType.MOTORCYCLE: 10, VehicleType.CAR: 20, VehicleType.TRUCK: 30}
    
    def __init__(self, name: str, floors: int = 3):
        self.name = name
        self.floors = [ParkingFloor(i + 1) for i in range(floors)]
        self.active_tickets = {}   # license_plate → Ticket
        self.lock = threading.Lock()
    
    def park(self, vehicle: Vehicle) -> Ticket:
        with self.lock:
            if vehicle.license_plate in self.active_tickets:
                raise ValueError(f"{vehicle} already parked!")
            
            for floor in self.floors:
                spot = floor.find_spot(vehicle)
                if spot:
                    spot.park(vehicle)
                    ticket = Ticket(vehicle, spot)
                    self.active_tickets[vehicle.license_plate] = ticket
                    return ticket
            
            raise ValueError("Parking lot full for this vehicle type!")
    
    def unpark(self, license_plate: str) -> float:
        with self.lock:
            if license_plate not in self.active_tickets:
                raise ValueError(f"No active ticket for {license_plate}")
            
            ticket = self.active_tickets.pop(license_plate)
            ticket.exit_time = datetime.now()
            ticket.spot.unpark()
            
            # Calculate fee
            duration = (ticket.exit_time - ticket.entry_time).total_seconds() / 3600
            hours = max(1, int(duration) + (1 if duration % 1 > 0 else 0))  # ceiling
            rate = self.RATES[ticket.vehicle.vehicle_type]
            fee = hours * rate
            return fee
    
    def status(self):
        print(f"\n🅿️ {self.name} — Status")
        print(f"  Active vehicles: {len(self.active_tickets)}")
        for floor in self.floors:
            avail = floor.available_count()
            print(f"  Floor {floor.floor_num}: {avail}")

# --- Demo ---
lot = ParkingLot("City Center Parking", floors=2)
lot.status()

# Park vehicles
t1 = lot.park(Car("KA-01-1234"))
t2 = lot.park(Motorcycle("KA-02-5678"))
t3 = lot.park(Truck("KA-03-9999"))
print(f"\nTickets: {t1}, {t2}, {t3}")

lot.status()

# Unpark
fee = lot.unpark("KA-01-1234")
print(f"\nFee for KA-01-1234: ₹{fee} (minimum 1 hour)")
lot.status()

In [ ]:
# ============================================================
# 4. Design Decisions & Extensions
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  DESIGN DECISIONS TO DISCUSS IN INTERVIEW                    ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  1. Thread Safety                                             ║
║     • Lock on park/unpark (used threading.Lock)              ║
║     • Could use per-floor locks for finer granularity        ║
║                                                                ║
║  2. Finding Available Spots — O(1)                           ║
║     • Current: O(N) scan through spots                       ║
║     • Better: maintain a heap/set of available spots per size ║
║     • available_small = {spot1, spot7, spot12}               ║
║     • park: pop from set, unpark: add back to set            ║
║                                                                ║
║  3. Payment Strategy (Strategy Pattern)                      ║
║     • HourlyRate, FlatRate, DynamicRate                      ║
║     • Different for weekdays vs weekends                     ║
║                                                                ║
║  4. Entry/Exit Gates                                         ║
║     • Multiple gates → need distributed lock or queue        ║
║     • Display boards showing availability per floor          ║
║                                                                ║
║  5. Extensions                                               ║
║     • Electric vehicle charging spots                        ║
║     • Handicapped spots (near entrance)                      ║
║     • Valet parking (separate queue)                         ║
║     • Reservation system                                     ║
║     • License plate recognition (no ticket needed)           ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design a parking lot system | Vehicle/Spot enums, ParkingLot/Floor/Spot/Ticket classes, thread-safe |
| 2 | How to find spots in O(1)? | Maintain set/heap of available spots per SpotSize |
| 3 | How to handle concurrency? | Lock on park/unpark. Per-floor locks for scalability |
| 4 | How to support different pricing? | Strategy pattern: HourlyRate, FlatRate, DynamicPricing |
| 5 | How would you scale this? | Multiple gates, distributed locks, display boards via pub/sub |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Start with requirements → class diagram → code        │
## │  • Enums for VehicleType, SpotSize                       │
## │  • Composition: Lot has Floors has Spots                 │
## │  • Thread-safe: Lock on park/unpark                      │
## │  • Discuss trade-offs and extensions proactively          │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **LRU Cache Design**